# Semana 4 · Parte 3 — Funciones singulares y transformaciones

**Señales y Sistemas** · 2026-2
Universidad Católica San Pablo · Facultad de Ingeniería
Ing. Electrónica y de Telecomunicaciones
Jefe de práctica: Ing. Jeanpier Ancori Sánchez

---

## Objetivos

Al terminar esta sesión el alumno será capaz de:

1. Definir y graficar el escalón unitario, la rampa y el pulso rectangular.
2. Construir una señal definida por tramos combinando funciones singulares.
3. Aplicar las cuatro transformaciones de la variable independiente —desplazamiento,
   escalado, reflexión y su combinación— y predecir el resultado antes de graficarlo.
4. Distinguir el escalado en tiempo continuo del **diezmado** y la **interpolación**
   en tiempo discreto.

> **Requisito previo:** Partes 1 y 2. Aquí se dan por conocidos el vector de tiempo,
> `plot`, `stem`, `subplot` y la notación $x[n] = x(nT_s)$.

---
## 1. Preparación del entorno

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["axes.grid"] = True

---
## 2. El escalón unitario

El **escalón unitario** $u(t)$ vale cero antes del origen y uno a partir de él:

$$
\boxed{
u(t) =
\begin{cases}
0, & t < 0\\
1, & t > 0
\end{cases}
}
$$

Es la función singular más importante del curso: representa el instante en que se conecta
una fuente, se cierra un interruptor o arranca una excitación.

Desplazarlo consiste en cambiar el argumento. **El escalón sube donde su argumento se
anula**, lo cual conviene recordar porque el signo engaña:

$$
u(t - t_0) \ \text{sube en } t = t_0
\qquad\qquad
u(t + t_0) \ \text{sube en } t = -t_0
$$

### 2.1. `np.heaviside` y el valor en el origen

En NumPy el escalón es `np.heaviside(x, valor_en_cero)`. **El segundo argumento no es un
desplazamiento**: es el valor que toma la función exactamente en $x = 0$, donde el escalón
matemático no está definido.

| Llamada | Vale en $t=0$ | Convención |
|---|---|---|
| `np.heaviside(t, 0)` | 0 | continua por la derecha |
| `np.heaviside(t, 1)` | 1 | continua por la izquierda |
| `np.heaviside(t, 0.5)` | 0,5 | valor medio, la más usada en teoría de señales |

Confundir ese argumento con un desplazamiento es un error frecuente: escribir
`np.heaviside(t, 2)` **no** desplaza nada — deja un pico aislado de valor 2 en el origen,
que estropea cualquier suma posterior.

En este material se fija `0.5` dentro de la función `escalon`, y el desplazamiento se hace
siempre **sobre el argumento**: `escalon(t - t0)`.

In [ ]:
def escalon(t):
    """Escalón unitario u(t), con valor 0,5 en el origen."""
    u = np.heaviside(t, 0.5)
    return u


t = np.linspace(-3, 3, 601)

plt.figure(figsize=(12, 4))

# --- El desplazamiento va en el argumento ----------------------------
plt.subplot(1, 2, 1)
plt.plot(t, escalon(t), label="u(t)")
plt.plot(t, escalon(t - 1), label="u(t-1), sube en t = 1")
plt.plot(t, escalon(t + 1), label="u(t+1), sube en t = -1")
plt.title("El escalón sube donde su argumento se anula")
plt.xlabel("t [s]")
plt.ylabel("Amplitud")
plt.ylim(-0.2, 1.4)
plt.legend(loc="lower right")

# --- El segundo argumento NO es un desplazamiento --------------------
t_corto = np.linspace(-1, 1, 21)

plt.subplot(1, 2, 2)
plt.plot(t_corto, np.heaviside(t_corto, 0.5), "o-", label="valor en cero = 0,5")
plt.plot(t_corto, np.heaviside(t_corto, 2), "x--", label="valor en cero = 2 (¡pico!)")
plt.title("El segundo argumento no desplaza nada")
plt.xlabel("t [s]")
plt.ylabel("Amplitud")
plt.legend(loc="upper left")

plt.tight_layout()
plt.show()

---
## 3. La rampa

La **rampa unitaria** crece linealmente a partir del origen y es nula antes:

$$
\boxed{
r(t) = t\,u(t) =
\begin{cases}
0, & t < 0\\
t, & t \ge 0
\end{cases}
}
$$

De la definición se sigue una relación que reaparecerá al estudiar sistemas: la rampa es la
**integral** del escalón, y el escalón es la **derivada** de la rampa.

$$
r(t) = \int_{-\infty}^{t} u(\tau)\,d\tau
\qquad\qquad
\frac{d\,r(t)}{dt} = u(t)
$$

Una rampa general de pendiente $m$ que arranca en $t_0$ se escribe:

$$
\boxed{r(t) = m\,(t - t_0)\,u(t - t_0)}
$$

Multiplicar por $u(t-t_0)$ es lo que anula la recta antes de $t_0$. Ese producto es la forma
correcta de construirla: recortar con `np.maximum` solo funciona si la pendiente es positiva
y el mínimo es cero, y falla en cuanto se cambia el signo o se desplaza la recta.

In [ ]:
def rampa(t, pendiente=1.0, inicio=0.0):
    """Rampa de la pendiente dada que arranca en t = inicio y es nula antes."""
    r = pendiente * (t - inicio) * escalon(t - inicio)
    return r


t = np.linspace(-5, 5, 1001)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(t, rampa(t), label="r(t) = t u(t)")
plt.plot(t, escalon(t), "--", label="u(t)")
plt.title("La rampa es la integral del escalón")
plt.xlabel("t [s]")
plt.ylabel("Amplitud")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(t, rampa(t, pendiente=1, inicio=0), label="m = 1, arranca en t = 0")
plt.plot(t, rampa(t, pendiente=1/3, inicio=-3), label="m = 1/3, arranca en t = -3")
plt.plot(t, rampa(t, pendiente=-1, inicio=1), label="m = -1, arranca en t = 1")
plt.title("Rampas de distinta pendiente y distinto origen")
plt.xlabel("t [s]")
plt.ylabel("Amplitud")
plt.legend()

plt.tight_layout()
plt.show()

---
## 4. El pulso rectangular

Un **pulso rectangular** que vale 1 entre $a$ y $b$, y cero fuera, es la diferencia de dos
escalones:

$$
\boxed{p(t) = u(t - a) - u(t - b)}
$$

La idea es directa: el primer escalón "enciende" la señal en $t=a$ y el segundo la "apaga"
en $t=b$. Su anchura es $b - a$.

In [ ]:
def pulso(t, a, b):
    """Pulso rectangular unitario entre t = a y t = b."""
    p = escalon(t - a) - escalon(t - b)
    return p


t = np.linspace(-5, 5, 1001)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(t, escalon(t + 2), "--", label="u(t+2)")
plt.plot(t, escalon(t - 2), "--", label="u(t-2)")
plt.plot(t, pulso(t, -2, 2), linewidth=3, label="u(t+2) - u(t-2)")
plt.title("Un pulso es la diferencia de dos escalones")
plt.xlabel("t [s]")
plt.ylabel("Amplitud")
plt.ylim(-0.2, 1.4)
plt.legend(loc="lower right")

plt.subplot(1, 2, 2)
plt.plot(t, pulso(t, -3, -1), label="entre -3 y -1, anchura 2")
plt.plot(t, pulso(t, -1, 2), label="entre -1 y 2, anchura 3")
plt.plot(t, pulso(t, 2, 4), label="entre 2 y 4, anchura 2")
plt.title("Pulsos de distinta posición y anchura")
plt.xlabel("t [s]")
plt.ylabel("Amplitud")
plt.ylim(-0.2, 1.4)
plt.legend()

plt.tight_layout()
plt.show()

### 4.1. Señales definidas por tramos

Combinando rampas se construye cualquier señal formada por segmentos rectos. El
procedimiento es siempre el mismo: **recorrer los quiebres de izquierda a derecha y, en
cada uno, sumar la rampa que corrige la pendiente**.

Si en un quiebre la pendiente pasa de $m_{\text{antes}}$ a $m_{\text{después}}$, hay que
sumar una rampa de pendiente $m_{\text{después}} - m_{\text{antes}}$ que arranque ahí.

Construyamos un trapecio: sube con pendiente $+1$ desde $t=-2$, se mantiene plano en 2
desde $t=0$, y baja con pendiente $-1$ desde $t=1$ hasta anularse en $t=3$.

| Quiebre | Pendiente antes | Pendiente después | Rampa que se suma |
|---|---|---|---|
| $t=-2$ | 0 | +1 | `rampa(t, +1, -2)` |
| $t=0$ | +1 | 0 | `rampa(t, -1, 0)` |
| $t=1$ | 0 | −1 | `rampa(t, -1, 1)` |
| $t=3$ | −1 | 0 | `rampa(t, +1, 3)` |

In [ ]:
t = np.linspace(-5, 6, 1101)

trapecio = rampa(t, +1, -2) + rampa(t, -1, 0) + rampa(t, -1, 1) + rampa(t, +1, 3)

plt.figure(figsize=(10, 4))
plt.plot(t, trapecio, linewidth=2)

# Marcas verticales en los cuatro quiebres
plt.axvline(-2, color="gray", linestyle=":")
plt.axvline(0, color="gray", linestyle=":")
plt.axvline(1, color="gray", linestyle=":")
plt.axvline(3, color="gray", linestyle=":")

plt.title("Señal trapezoidal construida como suma de cuatro rampas")
plt.xlabel("t [s]")
plt.ylabel("Amplitud [V]")
plt.show()

---
## 5. Transformaciones de la variable independiente

Todas las transformaciones de esta sección actúan sobre el **argumento** de la señal, no
sobre su valor. La forma general es:

$$
\boxed{y(t) = x(a\,t + b)}
$$

donde $a$ produce el escalado —y la reflexión, si es negativo— y $b$ el desplazamiento.

Trabajaremos sobre una señal **asimétrica**, para que se distinga a simple vista si se ha
desplazado, comprimido o reflejado. Con una senoide, muchas de estas transformaciones son
indistinguibles unas de otras.

In [ ]:
def x(t):
    """
    Señal de referencia: triángulo asimétrico de 2 s de duración.

    Sube con pendiente +2 desde t = 0 hasta el pico de valor 1 en t = 0,5,
    y baja con pendiente -2/3 hasta anularse en t = 2.
    """
    y = rampa(t, 2, 0) + rampa(t, -2 - 2/3, 0.5) + rampa(t, 2/3, 2)
    return y


t = np.linspace(-4, 4, 1601)

plt.figure(figsize=(10, 4))
plt.plot(t, x(t), linewidth=2)
plt.title("Señal de referencia x(t)")
plt.xlabel("t [s]")
plt.ylabel("Amplitud [V]")
plt.show()

### 5.1. Desplazamiento

$$
\boxed{y(t) = x(t - t_0)}
$$

Con $t_0 > 0$ la señal se **retrasa**: se mueve hacia la derecha. Con $t_0 < 0$ se adelanta.

El signo despista, y conviene el mismo razonamiento que con el escalón: $y(t)$ toma en
$t = t_0$ el valor que $x$ tenía en $0$, así que el suceso ocurre $t_0$ segundos **más tarde**.

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(t, x(t), linewidth=2, label="x(t)")
plt.plot(t, x(t - 1), "--", linewidth=2, label="x(t-1), retrasada 1 s")
plt.plot(t, x(t + 1), ":", linewidth=2, label="x(t+1), adelantada 1 s")
plt.title("Desplazamiento en el tiempo")
plt.xlabel("t [s]")
plt.ylabel("Amplitud [V]")
plt.legend()
plt.show()

### 5.2. Escalado: compresión y expansión

$$
\boxed{y(t) = x(a\,t)}, \qquad a > 0
$$

| Valor de $a$ | Efecto | Duración |
|---|---|---|
| $a > 1$ | **compresión**: la señal ocurre más rápido | se divide entre $a$ |
| $0 < a < 1$ | **expansión**: la señal ocurre más lento | se multiplica por $1/a$ |

Otra vez el resultado es el contrario del que sugiere la intuición: **multiplicar** el tiempo
por 2 **comprime** la señal a la mitad. La razón es que $y(t)$ alcanza en $t=1$ el valor que
$x$ tenía en $t=2$, o sea que todo ocurre antes.

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(t, x(t), linewidth=2, label="x(t), dura 2 s")
plt.plot(t, x(2 * t), "--", linewidth=2, label="x(2t), comprimida, dura 1 s")
plt.plot(t, x(0.5 * t), ":", linewidth=2, label="x(t/2), expandida, dura 4 s")
plt.title("Escalado en el tiempo: compresión y expansión")
plt.xlabel("t [s]")
plt.ylabel("Amplitud [V]")
plt.legend()
plt.show()

### 5.3. Reflexión

$$
\boxed{y(t) = x(-t)}
$$

La señal se refleja respecto al eje vertical: lo que ocurría en $t$ pasa a ocurrir en $-t$.
Es el caso $a = -1$ de la forma general.

Esta es exactamente la operación que aparece en la definición de la convolución, más
adelante en el curso.

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(t, x(t), linewidth=2, label="x(t)")
plt.plot(t, x(-t), "--", linewidth=2, label="x(-t), reflejada")
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Reflexión respecto al eje vertical")
plt.xlabel("t [s]")
plt.ylabel("Amplitud [V]")
plt.legend()
plt.show()

### 5.4. Transformación combinada y orden de las operaciones

$$
\boxed{y(t) = x(a\,t + b)}
$$

Cuando se aplican escalado y desplazamiento a la vez, **el orden importa**. La forma segura
de resolverlo es factorizar el argumento:

$$
x(a\,t + b) = x\!\left(a\left(t + \frac{b}{a}\right)\right)
$$

leyéndolo así: primero se escala por $a$, después se desplaza $b/a$ — **no** $b$.

Comprobémoslo con $y(t) = x(2t - 2)$. Factorizando queda $x\big(2(t-1)\big)$: se comprime por
2 y se retrasa **1 s**, no 2 s. El panel de la derecha muestra qué sale si no se factoriza.

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(t, x(t), linewidth=2, label="x(t)")
plt.plot(t, x(2 * t - 2), "--", linewidth=2, label="y(t) = x(2t-2)")
plt.title("x(2t-2) = x(2(t-1)): comprimir y retrasar 1 s")
plt.xlabel("t [s]")
plt.ylabel("Amplitud [V]")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(t, x(2 * t - 2), linewidth=2, label="correcto: x(2(t-1))")
plt.plot(t, x(2 * (t - 2)), "--", linewidth=2, label="incorrecto: retrasar 2 s")
plt.title("Qué pasa si no se factoriza el argumento")
plt.xlabel("t [s]")
plt.ylabel("Amplitud [V]")
plt.legend()

plt.tight_layout()
plt.show()

---
## 6. Diezmado e interpolación

En **tiempo discreto** el escalado no puede hacerse con cualquier factor: el índice $n$ es
entero, así que solo tienen sentido los factores enteros.

**Diezmado** por un factor entero $M$ — se conserva una de cada $M$ muestras:

$$
\boxed{x_d[n] = x[n\,M]}
$$

La señal resultante tiene $M$ veces menos muestras, y la frecuencia de muestreo efectiva baja
a $f_m / M$. Es el análogo discreto de la compresión.

**Interpolación** por un factor entero $L$ — se insertan $L-1$ muestras entre cada par. La
señal resultante tiene $L$ veces más muestras y $f_m$ sube a $L\,f_m$. Es el análogo discreto
de la expansión.

> **La confusión más común:** diezmar **no** es evaluar la señal en menos puntos de un
> `linspace`. Se parte de una señal ya muestreada, $x[n]$, y se seleccionan sus muestras
> **por índice**. Por eso todas las gráficas de esta sección usan `stem` frente al índice $n$,
> y no `plot` frente a $t$.

En Python, quedarse con una de cada $M$ muestras se escribe `x_n[::M]`. Los dos puntos
significan «todo el arreglo» y el tercer número es el paso.

In [ ]:
# Señal discreta de partida: senoide de 1 Hz muestreada a 32 Hz durante 1 s
f = 1.0
fm = 32.0
N = 32

n = np.arange(N)
x_n = np.sin(2 * np.pi * f * n / fm)

M = 4                        # factor de diezmado

x_diezmada = x_n[::M]        # una de cada M muestras
n_diezmada = n[::M]          # y los índices que les corresponden

print("Señal original:", len(x_n), "muestras a", fm, "Hz")
print("Diezmada por", M, ":", len(x_diezmada), "muestras a", fm / M, "Hz")

plt.figure(figsize=(10, 6))

plt.subplot(2, 1, 1)
plt.stem(n, x_n)
plt.title(f"x[n] original: {len(x_n)} muestras a {fm} Hz")
plt.ylabel("Amplitud")

plt.subplot(2, 1, 2)
plt.stem(n_diezmada, x_diezmada)
plt.title(f"Diezmada por M = {M}: {len(x_diezmada)} muestras a {fm/M} Hz")
plt.xlabel("Índice n de la señal original")
plt.ylabel("Amplitud")

plt.tight_layout()
plt.show()

Ahora el camino inverso: partimos de la señal diezmada e intentamos recuperar la resolución
original insertando muestras intermedias.

`np.interp(donde_quiero, donde_tengo, valores)` calcula los valores intermedios por
interpolación lineal, es decir uniendo con rectas las muestras que sí conocemos.

In [ ]:
L = 4                          # factor de interpolación

n_conocidos = n_diezmada                  # índices donde sí hay muestra: 0, 4, 8, ...
n_todos = np.arange(N)                    # índices donde queremos tenerla: 0, 1, 2, ...

x_interpolada = np.interp(n_todos, n_conocidos, x_diezmada)

print("Diezmada   :", len(x_diezmada), "muestras")
print("Interpolada:", len(x_interpolada), "muestras")

plt.figure(figsize=(10, 6))

plt.subplot(2, 1, 1)
plt.stem(n_diezmada, x_diezmada)
plt.title(f"Punto de partida: la señal diezmada, {len(x_diezmada)} muestras")
plt.ylabel("Amplitud")

plt.subplot(2, 1, 2)
plt.stem(n_todos, x_interpolada)
plt.title(f"Interpolada por L = {L}: {len(x_interpolada)} muestras")
plt.xlabel("Índice n")
plt.ylabel("Amplitud")

plt.tight_layout()
plt.show()

La comparación final deja ver el punto importante: **diezmar destruye información, y la
interpolación no la devuelve**. La señal reconstruida se parece a la original, pero no
coincide con ella; el error crece con el factor de diezmado.

In [ ]:
error = x_interpolada - x_n

print("Error máximo:", np.max(np.abs(error)))

plt.figure(figsize=(10, 4))
plt.plot(n, x_n, "o-", label="original x[n]")
plt.plot(n_todos, x_interpolada, "s--", label="diezmada e interpolada")
plt.plot(n, error, ":", label="error")
plt.title(f"Diezmar por {M} e interpolar por {L} no recupera la señal")
plt.xlabel("Índice n")
plt.ylabel("Amplitud")
plt.legend()
plt.show()

---
## 7. Ejercicios propuestos

1. Construye con rampas la señal que vale 0 antes de $t=-1$, sube linealmente hasta 3 en
   $t=1$, se mantiene constante hasta $t=2$ y cae bruscamente a 0. Grafícala.

2. Para la señal de referencia $x(t)$ de la sección 5, dibuja **a mano** y luego verifica
   con código: $x(-2t+4)$. Indica el orden de las operaciones que has aplicado.

3. Escribe una función `pulso_centrado(t, centro, anchura)` que devuelva un pulso
   rectangular usando la función `pulso` que ya existe. Grafica tres pulsos distintos
   llamándola tres veces.

4. Demuestra numéricamente que la derivada de la rampa es el escalón, usando `np.gradient`
   sobre `rampa(t)`. ¿Por qué el resultado no es exacto en el origen?

5. Repite el diezmado de la sección 6 con $M = 2$, con $M = 8$ y con $M = 16$. ¿A partir de
   qué valor de $M$ la señal reconstruida deja de parecerse a la original? Relaciónalo con
   el número de muestras por período.